# 02 - Baseline inference (original pretrained CatVTON)

Runs the **unmodified pretrained** CatVTON weights (`zhengchong/CatVTON`, `vitonhd-16k-512` attention
weights on top of the Stable Diffusion inpainting UNet) on held-out VITON-HD test samples, shows
person / garment / result, and saves outputs plus the exact generation settings.

* Needs a GPU (run notebook 01 first) and the VITON-HD dataset in your Drive.
* Weights download into the runtime cache (a few GB); nothing is stored in the repository.
* If the GPU cannot load the model the notebook stops with the real error. It never fabricates results.
* License: CatVTON and its checkpoints are CC BY-NC-SA 4.0 (noncommercial).

In [ ]:
# --- Setup: Google Drive (persistent storage) + engine source -------------------------------
# Free runtimes disconnect. Everything that must survive (dataset, checkpoints, outputs) lives in Drive.
from google.colab import drive
drive.mount('/content/drive')

import pathlib, zipfile
WORK = pathlib.Path('/content/drive/MyDrive/virtualfit')
WORK.mkdir(parents=True, exist_ok=True)
ENGINE_ZIP = WORK / 'ai-engine.zip'   # create locally with:  python ai-engine/tools/package_engine.py
assert ENGINE_ZIP.exists(), f'Upload ai-engine.zip to {WORK} first (see ai-engine/README.md)'
if not pathlib.Path('/content/ai-engine').exists():
    zipfile.ZipFile(ENGINE_ZIP).extractall('/content')

%cd /content/ai-engine
!pip -q install -e . --no-deps
!pip -q install "diffusers>=0.31" "transformers>=4.46" "accelerate>=0.31" "peft>=0.13" safetensors lpips scikit-image opencv-python-headless pyyaml pydantic

In [ ]:
# --- Dataset location -------------------------------------------------------------------------
# VITON-HD is NOT included in this project and is licensed for research use only.
# Download it yourself from https://github.com/shadow2496/VITON-HD (link in its README) and place the
# unzipped folder at  <WORK>/VITON-HD  so that it contains train/, test/, train_pairs.txt, test_pairs.txt.
VITON_ROOT = WORK / 'VITON-HD'
assert (VITON_ROOT / 'train_pairs.txt').exists(), (
    f'{VITON_ROOT} is missing or incomplete. Download VITON-HD and put it there (see ai-engine/README.md).')
RUN = WORK / 'runs' / 'exp1'
DATA = RUN / 'data'

In [ ]:
# --- Validate the dataset and write train/val/held-out manifests (200 / 40 / 40) ----------------
# train+val come from the official TRAIN split; held-out comes from the official TEST split and is
# used for evaluation only. The command fails loudly if the dataset has too few valid samples.
!python -m virtualfit_engine.dataset.cli prepare --root "{VITON_ROOT}" --out "{DATA}" --n-train 200 --n-val 40 --n-heldout 40

In [ ]:
# --- Load the pretrained model ---------------------------------------------------------------
import json, time, torch
from pathlib import Path
from PIL import Image
from virtualfit_engine.dataset.viton_hd import read_manifest
from virtualfit_engine.models import EngineModels, ModelPaths, ModelUnavailable
from virtualfit_engine.training.data import load_record_mask

paths = ModelPaths()                      # pretrained only: no adapter
models = EngineModels(paths)
try:
    pipe = models.pipeline('pretrained')
except ModelUnavailable as e:
    raise SystemExit(f'CANNOT RUN: [{e.code}] {e.message}')
except Exception as e:                    # e.g. CUDA OOM while loading
    raise SystemExit(f'MODEL FAILED TO LOAD: {type(e).__name__}: {e}')
print('loaded; VRAM allocated: %.2f GB' % (torch.cuda.memory_allocated() / 1024**3))

In [ ]:
# --- Baseline inference on held-out samples ----------------------------------------------------
import matplotlib.pyplot as plt
from virtualfit_engine.postprocessing import repaint
from virtualfit_engine.preprocessing import prepare_inputs

N, STEPS, GUIDANCE, SEED = 6, 30, 2.5, 555
heldout = read_manifest(DATA / 'heldout.jsonl')[:N]
OUT = RUN / 'baseline'; OUT.mkdir(parents=True, exist_ok=True)

rows, torch_peak = [], []
fig, axes = plt.subplots(len(heldout), 3, figsize=(9, 4.5 * len(heldout)))
for i, rec in enumerate(heldout):
    person = Image.open(VITON_ROOT / rec.person).convert('RGB')
    garment = Image.open(VITON_ROOT / rec.cloth).convert('RGB')
    mask = load_record_mask(VITON_ROOT, rec)
    torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
    gen = torch.Generator(device='cpu').manual_seed(SEED)
    result = pipe(person, garment, mask, STEPS, GUIDANCE, paths.height, paths.width, gen)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    prep = prepare_inputs(person, garment, mask, paths.width, paths.height)
    final = repaint(prep.person, prep.mask, result)
    final.save(OUT / f'{rec.id}.png')
    rows.append({'id': rec.id, 'latency_s': round(dt, 2), 'peak_gpu_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 2)})
    for ax, im, title in zip(axes[i], (prep.person, prep.garment, final), ('person', 'garment', 'baseline result')):
        ax.imshow(im); ax.set_title(f'{rec.id}: {title}'); ax.axis('off')
plt.tight_layout(); plt.show()
rows

In [ ]:
# --- Record exactly how these outputs were produced -----------------------------------------------
import diffusers, transformers, peft
settings = {
    'model': 'pretrained CatVTON (no adapter)', 'base_model': paths.base_model, 'vae': paths.vae_model,
    'attention_weights': f'{paths.attn_repo}/{paths.attn_version}', 'resolution_hw': [paths.height, paths.width],
    'steps': STEPS, 'guidance_scale': GUIDANCE, 'seed': SEED, 'dtype': 'float16', 'repaint': True,
    'samples': [r.id for r in heldout], 'per_sample': rows,
    'gpu': torch.cuda.get_device_name(0),
    'versions': {'torch': torch.__version__, 'diffusers': diffusers.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__},
}
(OUT / 'baseline_settings.json').write_text(json.dumps(settings, indent=2))
print('saved to', OUT)

## Optional: parity check against the reference implementation

Our sampler (`virtualfit_engine/pipeline.py`) is a port of CatVTON's. On tiny random models it matches the
reference to within 1 pixel (unit tests). This cell repeats the comparison with the **real weights** on
one sample, so you know the engine reproduces the official baseline. It clones CatVTON at the commit this
project was developed against (`7818397f25`).

In [ ]:
PARITY = True
if PARITY:
    !git clone -q https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON && git -C /content/CatVTON checkout -q 7818397f25613beedb3d861a34769f607cfcf3b1
    !pip -q install -r /content/CatVTON/requirements.txt 2>/dev/null || true
    import sys, numpy as np
    sys.path.insert(0, '/content/CatVTON')
    from model.pipeline import CatVTONPipeline
    ref = CatVTONPipeline(base_ckpt=paths.base_model, attn_ckpt=paths.attn_repo, attn_ckpt_version=paths.attn_version,
                          weight_dtype=torch.float16, device='cuda', skip_safety_check=True)
    rec = heldout[0]
    person = Image.open(VITON_ROOT / rec.person).convert('RGB'); garment = Image.open(VITON_ROOT / rec.cloth).convert('RGB')
    mask = load_record_mask(VITON_ROOT, rec)
    torch.manual_seed(1); ours = pipe(person, garment, mask, STEPS, GUIDANCE, paths.height, paths.width, torch.Generator(device='cpu').manual_seed(SEED))
    torch.manual_seed(1); theirs = ref(person, garment, mask, num_inference_steps=STEPS, guidance_scale=GUIDANCE, height=paths.height, width=paths.width, generator=torch.Generator(device='cpu').manual_seed(SEED))[0]
    d = np.abs(np.asarray(ours, np.int16) - np.asarray(theirs, np.int16))
    print('max abs pixel diff:', int(d.max()), '| mean:', round(float(d.mean()), 4), '(fp16 kernels may differ slightly; large values would indicate a porting bug)')